<a href="https://colab.research.google.com/github/bashaierr04/LLM-finetuning-CIS492/blob/main/bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install transformers[torch] datasets pandas scikit-learn

import pandas as pd
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import pipeline

train_df = pd.read_csv('/content/testfinal.csv')
test_df = pd.read_csv('/content/testfinal.csv')
val_df = pd.read_csv('/content/valfinal.csv')


model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["clean_review"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
)

# 6. تشغيل الـ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("train time")
trainer.train()


#final result
print("final result")
print(trainer.evaluate(test_ds))

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)
#test any sentence
result = classifier(" the food was very good delivery was bad but i will give the resturant another chance ")
print(result)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

final result


{'eval_loss': 0.4750853180885315, 'eval_runtime': 3.1464, 'eval_samples_per_second': 146.198, 'eval_steps_per_second': 18.434, 'epoch': 3.0}
[{'label': 'LABEL_1', 'score': 0.5553107857704163}]


# New section

In [ ]:
from transformers import pipeline

# إنشاء "أنبوب" جاهز للتحليل باستخدام الموديل والتوكنايزر اللي دربناهم
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

# اختبر أي جملة فوراً
result = classifier("the food was good and delivery bad")
print(result)

In [ ]:
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

# اختبر أي جملة فوراً
result = classifier(" the food was GOOD")
print(result)

[{'label': 'LABEL_2', 'score': 0.7533696889877319}]
